## Aeropulse — Gold Helpers

**Purpose:** Shared functions reused across the gold dimension and fact notebooks.

**Used by:** `dim-airport`, `dim-airport-destination`, `dim-carrier`, `dim-date`, `fact-flight` (via `%run gold-helper`)

**Defines:**
- `add_sk_key(df, key_columns, sk_column_name)` — deterministic SHA-256 surrogate key
- `write_to_gold(input_df, target_table, merge_condition, columns_to_update)` — create-or-merge write into the gold Delta table, stamping `created_timestamp` / `updated_timestamp`


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
# generate surrogate key
def add_sk_key(df, key_columns, sk_column_name = "sk"):
    return df.withColumn(
        sk_column_name,
        F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_columns]), 256)
    )

In [ ]:
def write_to_gold(
    input_df,
    target_table,
    merge_condition,
    columns_to_update
):
    """
    create the Delta table if it does not exists.
    otherwise merges the input dataframe into the target table.
    """
    final_df = input_df.withColumn("created_timestamp", F.current_timestamp()).withColumn("updated_timestamp", F.current_timestamp())

    if not spark.catalog.tableExists(target_table):
        final_df.write.format('delta').mode('overwrite').saveAsTable(target_table)
    else:
        delta_table = DeltaTable.forName(spark, target_table)
        update_map = {column: f"s.{column}" for column in columns_to_update}
        update_map["updated_timestamp"] = "s.updated_timestamp"
        (
            delta_table.alias("t")
            .merge(
                final_df.alias("s"),
                merge_condition
            )
            .whenMatchedUpdate(
                set=update_map
            )
            .whenNotMatchedInsertAll()
            .execute()
        )